<a href="https://colab.research.google.com/github/fleurclochette/Scientific-Project-SS26/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import subprocess
import pandas as pd
import yfinance as yf
from google.colab import userdata

# Securely fetching credentials from Colab Secrets
GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = userdata.get('GITHUB_USER')
GH_EMAIL = "tushar.latiyan@st.ovgu.de"
REPO_URL = f"https://{GH_TOKEN}@github.com/fleurclochette/Scientific-Project-SS26.git"

# Defining the exact 23-asset universe
tickers = [
    '^GSPC', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY',
    '^AXJO', '^DJI', '^FCHI', '^FTSE', '^GDAXI', '^GSPTSE', '^HSI',
    '^IXIC', '^KS11', '^N225', '^RUT', '^STOXX50E', '^TWII'
]

print("Pulling market price data from Jan 2000 through July 2026...")

# Downloading raw price data (auto_adjust=False preserves standard Adj Close)
df_raw = yf.download(tickers, start='2000-01-01', end='2026-08-01', auto_adjust=False)['Adj Close']

# Cleaning data (Forward-fill holiday blanks; drop only pure weekend rows)
prices = df_raw.ffill().dropna(how='all')

# CROSS-CHECK: Verify the time frame and dimensions explicitly
start_date = prices.index.min().date()
end_date = prices.index.max().date()
print(f"\n--- DATA VALIDATION ---")
print(f"First Trading Day: {start_date}")
print(f"Last Trading Day:  {end_date}")

if str(start_date)[:4] == '2000':
    print("Start date check passed: Data begins in 2000.")
else:
    print(f"WARNING: Data begins in {start_date}.")

if str(end_date) == '2026-07-31':
    print("End date check passed: Data ends exactly on July 31, 2026.")
else:
    print(f"WARNING: End date is {end_date}, not July 31, 2026.")

print(f"Total Rows (Prices):     {len(prices)}")
print(f"Total Columns (Assets):  {len(prices.columns)}")
print(f"-----------------------\n")

# Saving the price file locally
os.makedirs('data/raw', exist_ok=True)
prices_path = 'data/raw/master_23_assets_prices_2000_2026.csv'
prices.to_csv(prices_path)
print("Saved raw prices locally.")

# Pushing prices to GitHub and remove the returns CSV
print("Updating repository on GitHub...")
cmds = [
    "rm -rf temp_repo",
    f"git clone {REPO_URL} temp_repo",
    "mkdir -p temp_repo/data/raw",
    f"cp {prices_path} temp_repo/data/raw/",
    "rm -f temp_repo/data/raw/master_23_assets_returns_2000_2026.csv",
    f'cd temp_repo && git config user.email "{GH_EMAIL}" && git config user.name "{GH_USER}"',
    'cd temp_repo && git add -A data/raw/',
    'cd temp_repo && git commit -m "chore: retain only raw price data (2000-2026) and remove returns" || true',
    'cd temp_repo && git push origin main',
    "rm -rf temp_repo"
]

for cmd in cmds:
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Git update complete: master price file is saved, and returns file has been removed.")

[****                   9%                       ]  2 of 23 completed

Pulling market price data from Jan 2000 through July 2026...


[*********************100%***********************]  23 of 23 completed



--- DATA VALIDATION ---
First Trading Day: 2000-01-03
Last Trading Day:  2026-07-31
Start date check passed: Data begins in 2000.
End date check passed: Data ends exactly on July 31, 2026.
Total Rows (Prices):     6916
Total Columns (Assets):  23
-----------------------

Saved raw prices locally.
Updating repository on GitHub...
Git update complete: master price file is saved, and returns file has been removed.
